In [ ]:
from google.colab import userdata
userdata.get('google_api')

In [1]:
!pip install transformers peft datasets langchain langchain-groq langchain-google-genai accelerate bitsandbytes --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 11.0 MB/s eta 0:00:00


In [4]:
!pip install rank-bm25 --quiet

In [8]:
import getpass
import os

api_key = getpass.getpass("Enter your Google API Key: ")

Enter your Google API Key: ··········


In [9]:
api_key = api_key.strip()

### Part 1: Document Corpus setup

10 technical documents related to ML and NLP. 1 document with proper noun for BM25 - dense search comparison

In [3]:
corpus = [
    # topic: attention mechanisms
    "The Transformer architecture relies on the Self-Attention mechanism to weigh the importance of different words in a sequence.",
    "Attention mechanisms allow models to focus on specific parts of the input, improving performance on long-range dependencies.",
    "Cross-attention is a variant where the queries come from one sequence and keys/values come from another, often used in encoder-decoder setups.",

    # topic: optimization
    "Stochastic Gradient Descent (SGD) is an iterative method for optimizing an objective function with suitable smoothness properties.",
    "Adam is an optimization algorithm that can be used instead of the classical SGD procedure to update network weights iteratively.",
    "Learning rate scheduling involves adjusting the step size during training to improve convergence and avoid local minima.",

    # topic: tree based models
    "The Gini impurity is a measure of how often a randomly chosen element from the set would be incorrectly labeled.",
    "Decision trees use entropy or Gini impurity to decide where to split the data for optimal classification.",
    "Random Forests are ensemble learning methods that operate by constructing a multitude of decision trees at training time.",

    # technical jargon
    "The RoBERTa model improves upon BERT by removing the next-sentence prediction objective and training with much larger mini-batches."
]

print(f"Corpus size: {len(corpus)} documents")

Corpus size: 10 documents


### Part 2: Hybrid Retriever Implementation (BM25 + SBERT + RRF)

In [5]:
import numpy as np
import pandas as pd
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, util

class HybridRetriever:
    def __init__(self, corpus: list[str], k: int = 60):
        self.corpus = corpus
        self.k = k

        # Initialize BM25 (Lexical)
        # Tokenize on whitespace as suggested in the hints
        self.tokenized_corpus = [doc.lower().split() for doc in corpus]
        self.bm25 = BM25Okapi(self.tokenized_corpus)

        # Initialize SBERT (Dense)
        self.encoder = SentenceTransformer('all-MiniLM-L6-v2')
        self.corpus_embeddings = self.encoder.encode(corpus, convert_to_tensor=True)

    def retrieve(self, query: str, top_k: int = 5) -> list[dict]:
        # BM25 Ranking
        tokenized_query = query.lower().split()
        bm25_scores = self.bm25.get_scores(tokenized_query)
        # Sort indices by score (descending) to get rank
        bm25_ranking = np.argsort(bm25_scores)[::-1]
        # Map doc_id to its rank (1-indexed)
        bm25_ranks = {doc_idx: rank + 1 for rank, doc_idx in enumerate(bm25_ranking)}

        # SBERT Ranking
        query_embedding = self.encoder.encode(query, convert_to_tensor=True)
        cos_scores = util.cos_sim(query_embedding, self.corpus_embeddings)[0]
        # Sort indices by score (descending) to get rank
        sbert_ranking = np.argsort(cos_scores.cpu().numpy())[::-1]
        # Map doc_id to its rank (1-indexed)
        sbert_ranks = {doc_idx: rank + 1 for rank, doc_idx in enumerate(sbert_ranking)}

        # Reciprocal Rank Fusion (RRF)
        combined_results = []
        for i in range(len(self.corpus)):
            # RRF formula: 1 / (k + rank)
            rrf_score = (1.0 / (self.k + bm25_ranks[i])) + (1.0 / (self.k + sbert_ranks[i]))

            combined_results.append({
                "doc_id": i,
                "rrf_score": rrf_score,
                "bm25_rank": bm25_ranks[i],
                "sbert_rank": sbert_ranks[i],
                "text": self.corpus[i]
            })

        # Sort by RRF score descending and return top_k
        combined_results.sort(key=lambda x: x["rrf_score"], reverse=True)
        return combined_results[:top_k]

# Initialize for testing
retriever = HybridRetriever(corpus)
test_results = retriever.retrieve("What is attention?")

pd.DataFrame(test_results)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

,doc_id,rrf_score,bm25_rank,sbert_rank,text
0,2,0.031754,4,2,Cross-attention is a variant where the queries...
1,6,0.031281,2,6,The Gini impurity is a measure of how often a ...
2,4,0.031258,3,5,Adam is an optimization algorithm that can be ...
3,1,0.030886,9,1,Attention mechanisms allow models to focus on ...
4,3,0.030886,1,9,Stochastic Gradient Descent (SGD) is an iterat...


### Part 3: Cross-Encoder Re-ranker
Accepts the original user query (not the HyDE-expanded version) as input and returns the top-k re-ranked documents with their cross-encoder scores


In [6]:
from sentence_transformers import CrossEncoder

# Initialize the Cross-Encoder model
rerank_model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

def rerank(query: str, candidates: list[dict], top_k: int = 3) -> list[dict]:
    """
    Re-ranks a list of candidate documents based on their relevance to the query.

    Args:
        query: The original user query.
        candidates: List of dicts from HybridRetriever.
        top_k: Number of documents to return after re-ranking.

    Returns:
        List of dicts with an added 'cross_score' field, sorted by that score.
    """
    # Prepare pairs: [query, document_text]
    pairs = [[query, cand['text']] for cand in candidates]

    # Predict relevance scores
    scores = rerank_model.predict(pairs)

    # Attach scores to the candidate dictionaries
    for i, score in enumerate(scores):
        candidates[i]['cross_score'] = float(score)

    # Sort candidates by the new cross-encoder score in descending order
    reranked_candidates = sorted(candidates, key=lambda x: x['cross_score'], reverse=True)

    return reranked_candidates[:top_k]

# Testing with candidates from Part 2
top_candidates = test_results # Results from the previous HybridRetriever test
reranked_results = rerank("What is attention?", top_candidates)

# Display re-ranked results
pd.DataFrame(reranked_results)

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

,doc_id,rrf_score,bm25_rank,sbert_rank,text,cross_score
0,1,0.030886,9,1,Attention mechanisms allow models to focus on ...,3.791545
1,2,0.031754,4,2,Cross-attention is a variant where the queries...,2.554087
2,3,0.030886,1,9,Stochastic Gradient Descent (SGD) is an iterat...,-9.383696


### Part 4: HyDE (Hypothetical Document Embedding) for Query Expansion
Gemini generates a hypothetical answer, which is uses as the retrieval query

In [13]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.0, google_api_key=api_key)

# Define the HyDE Prompt Template
hyde_prompt = ChatPromptTemplate.from_template("""
write a brief, technical 2-3 sentence paragraph answering the following question. Focus on using precise machine learning terminology and technical jargon.

Question: {query}

Technical Answer:
""")

# Construct the Chain
hyde_chain = hyde_prompt | llm | StrOutputParser()

def generate_hyde_doc(query: str) -> str:
    return hyde_chain.invoke({"query": query})


user_query = "what is attention?"
hyde_query = generate_hyde_doc(user_query)

print(f"Original Query: {user_query}")
print("-" * 30)
print(f"HyDE Expanded Query:\n{hyde_query}")

Original Query: what is attention?
------------------------------
HyDE Expanded Query:
Attention is a neural mechanism that computes a weighted sum of input "value" vectors, where the weights are dynamically derived from the compatibility between a "query" vector and corresponding "key" vectors. This process generates a context-aware representation for each output element, effectively capturing long-range dependencies and mitigating information bottlenecks inherent in fixed-size representations.


### Part 5: End-to-End Pipeline
Query Expansion → Hybrid Retrieval → Re-Ranking → LLM Generation

In [14]:
# Define the final RAG generation prompt
rag_prompt = ChatPromptTemplate.from_template("""
You are a university AI/ML assistant. Use the following pieces of retrieved context to answer the student's question.
If the answer is not in the context, say that you don't know. Use professional technical language.

Context:
{context}

Student Question: {query}

Final Technical Answer:
""")

# Construct the generation chain
rag_chain = rag_prompt | llm | StrOutputParser()

def advanced_rag(user_query: str) -> str:
    """
    Full pipeline: Query Expansion → Hybrid Retrieval → Re-Ranking → LLM Generation
    """
    # Query Expansion (HyDE)
    hyde_doc = generate_hyde_doc(user_query)

    # Hybrid Retrieval
    # Use the expanded HyDE query to search the corpus
    # top_k=10 to give the re-ranker a good selection
    initial_candidates = retriever.retrieve(hyde_doc, top_k=10)

    # Re-Ranking
    # Re-rank using the ORIGINAL user query for precision
    # Returns the top 3 most relevant documents
    final_candidates = rerank(user_query, initial_candidates, top_k=3)

    # Prepare Context for Generation
    context_text = "\n\n".join([f"Source {i+1}: {c['text']}" for i, c in enumerate(final_candidates)])

    # Final LLM Generation
    final_answer = rag_chain.invoke({
        "context": context_text,
        "query": user_query
    })

    return final_answer

# Test the full pipeline
print("Advanced RAG Pipeline Test")
response = advanced_rag("how do transformers encode meaning?")
print(response)

Advanced RAG Pipeline Test
Based on the provided context, the information on how Transformers encode meaning is limited.

The context states that "The Transformer architecture relies on the Self-Attention mechanism to weigh the importance of different words in a sequence." While Self-Attention is a fundamental component that processes relationships between words, the context does not explicitly detail the full mechanism by which Transformers encode meaning into a comprehensive representation.


### Part 6: Comparison Experiment

*   **Naïve RAG Retrieval**: Dense-only retrieval (SBERT cosine, no expansion, no re-ranking).
*   **Advanced RAG Retrieval**: HyDE (query expansion) → Hybrid Retrieval (BM25 + SBERT + RRF) → Cross-Encoder re-ranking (find the most relevant document)

In [15]:
def naive_rag_retrieve(query: str, top_k: int = 1) -> str:
    """
    Naïve RAG: Dense-only retrieval (SBERT cosine, no expansion, no re-ranking).
    Returns the text of the top-1 retrieved document.
    """
    query_embedding = retriever.encoder.encode(query, convert_to_tensor=True)
    cos_scores = util.cos_sim(query_embedding, retriever.corpus_embeddings)[0]
    top_doc_idx = np.argmax(cos_scores.cpu().numpy())
    return corpus[top_doc_idx]

def get_advanced_top_doc(query: str) -> str:
    """
    Extracts the top-1 document after the FULL Advanced RAG retrieval process
    (HyDE -> Hybrid -> Re-rank).
    """
    # HyDE Expansion
    hyde_doc = generate_hyde_doc(query)
    # Hybrid Retrieval
    candidates = retriever.retrieve(hyde_doc, top_k=10)
    # Re-ranking
    reranked = rerank(query, candidates, top_k=1)
    return reranked[0]['text']

# Test Queries
queries = [
    "how do transformers encode meaning?",
    "optimization techniques for training",
    "what is the RoBERTa model?"
]

# Run Experiment
results = []
for q in queries:
    naive_doc = naive_rag_retrieve(q)
    advanced_doc = get_advanced_top_doc(q)
    results.append({
        "Query": q,
        "Naïve RAG Top Doc": naive_doc,
        "Advanced RAG Top Doc": advanced_doc,
        "Different?": "Yes" if naive_doc != advanced_doc else "No"
    })

df_comparison = pd.DataFrame(results)
df_comparison

,Query,Naïve RAG Top Doc,Advanced RAG Top Doc,Different?
0,how do transformers encode meaning?,The Transformer architecture relies on the Sel...,The Transformer architecture relies on the Sel...,No
1,optimization techniques for training,Learning rate scheduling involves adjusting th...,Adam is an optimization algorithm that can be ...,Yes
2,what is the RoBERTa model?,The RoBERTa model improves upon BERT by removi...,The RoBERTa model improves upon BERT by removi...,No


|index|Query|Naïve RAG Top Doc|Advanced RAG Top Doc|Are they different?|
|---|---|---|---|---|
|0|how do transformers encode meaning?|The Transformer architecture relies on the Self-Attention<br>mechanism to weigh the importance of different words in a<br>sequence\.|The Transformer architecture relies on the Self-Attention<br>mechanism to weigh the importance of different words in a<br>sequence\.|No|
|1|optimization techniques for training|Learning rate scheduling involves adjusting the step size during<br>training to improve convergence and avoid local minima\.|Adam is an optimization algorithm that can be used instead<br>of the classical SGD procedure to update network weights<br>iteratively\.|Yes|
|2|what is the RoBERTa model?|The RoBERTa model improves upon BERT by removing the<br>next-sentence prediction objective and training with much larger<br>mini-batches\.|The RoBERTa model improves upon BERT by removing the<br>next-sentence prediction objective and training with much<br>larger mini-batches\.|No|